# Stage 2 Analysis

Stage 2 models weekly interruption risk after the learner has already passed the initial Stage 1 observation window. Each example uses the previous `SEQUENCE_LENGTH` weeks to predict whether the next labeled week is a dropout point.

Unlike Stage 1, this stage keeps the temporal structure of learner behavior. The model can therefore learn patterns such as sustained engagement, declining activity, irregular usage, or seasonal pauses instead of relying only on static early aggregates.


## Setup


In [3]:
import os
from pathlib import Path

os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ["OMP_NUM_THREADS"] = "1"

import numpy as np
import pandas as pd

import torch

try:
    import lightgbm as lgb
except ModuleNotFoundError:
    lgb = None

PROJECT_ROOT = Path.cwd()

from src.preprocess import build_stage2_dataset
from src import model_utils as lstm
from src import sequence_utils as seq
from src.metrics_utils import validation_tuned_binary_metrics

DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

RANDOM_STATE = 42
SEQUENCE_LENGTH = 12
BATCH_SIZE = 64
EPOCHS = 30
PATIENCE = 5
LEARNING_RATE = 1e-3

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)


device: cuda


## Build Dataset

The Stage 2 dataset is built with `build_stage2_dataset`. It creates one row per learner-week, with behavioral features such as activity, sessions, transactions, correctness, topic breadth, and calendar indicators.

The label `is_dropout_point` marks weeks that precede an observed interruption. It is a local sequence label, not a global statement that the learner permanently left the platform.


In [4]:
stage2 = build_stage2_dataset(DATA_DIR)
df_full = stage2.df_full
feature_cols = stage2.feature_cols
users_clean = stage2.users
user_features = stage2.user_features

print(f"timeline rows: {len(df_full):,}")
print(f"users: {df_full['user_id'].nunique():,}")
print(f"features: {len(feature_cols)}")
print(f"labeled rows: {df_full['is_dropout_point'].notna().sum():,}")
print(feature_cols)

df_full.head()


timeline rows: 408,377
users: 22,470
features: 17
labeled rows: 342,844
['n_events', 'n_active_days', 'mean_hour', 'n_click_events', 'n_view_events', 'n_sessions', 'n_topics_event', 'n_transactions', 'correct_rate', 'partial_rate', 'mean_evaluation_score', 'avg_response_time', 'n_documents', 'n_topics_transaction', 'activity_score', 'year', 'day']


,user_id,relative_week,n_events,n_active_days,mean_hour,n_click_events,n_view_events,n_sessions,n_topics_event,n_transactions,...,avg_response_time,n_documents,n_topics_transaction,activity_score,week_start,year,day,is_active,is_dropout_point,is_summer
0,387604,0,2.0,2.0,9.0,0.0,2.0,0.0,0.0,2.0,...,0.0,1.0,0.0,6.0,2021-05-22 05:12:11.249,2021,142,1,0.0,False
1,387604,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,2021-05-29 05:12:11.249,2021,149,0,0.0,False
2,387604,2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,2021-06-05 05:12:11.249,2021,156,0,0.0,False
3,387604,3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,2021-06-12 05:12:11.249,2021,163,0,0.0,False
4,387604,4,7.0,1.0,8.0,4.0,3.0,1.0,1.0,0.0,...,0.0,0.0,0.0,14.0,2021-06-19 05:12:11.249,2021,170,1,0.0,False


## Split And Scale

The split is performed by user, not by row. This avoids leakage where weeks from the same learner appear in both training and testing. Feature scaling is fit on the training users only, then applied to validation and test users.

This design makes the evaluation closer to the real task: predicting future behavior for learners not seen during training.


In [5]:
train_df, val_df, test_df, scaler = seq.split_and_scale_by_users(
    df_full,
    feature_cols,
    test_size=0.15,
    val_size=0.20,
    random_state=RANDOM_STATE,
)
print("train:", train_df.shape)
print("val:  ", val_df.shape)
print("test: ", test_df.shape)


train: (275745, 23)
val:   (70920, 23)
test:  (61712, 23)


## Sequence Datasets

Weekly rows are converted into fixed-length sequences. For each labeled week, the input is the previous `SEQUENCE_LENGTH` weeks and the target is the dropout-point label of the following week.

This representation lets the LSTM use the order of behavioral changes, which is precisely the information lost in a static aggregate table.


In [6]:
loaders, _ = lstm.make_sequence_loaders(
    train_df,
    val_df,
    test_df,
    feature_cols,
    sequence_length=SEQUENCE_LENGTH,
    batch_size=BATCH_SIZE,
)
train_loader, val_loader, test_loader = loaders
print("sequence batches:", len(train_loader), len(val_loader), len(test_loader))


sequence batches: 2250 591 506


## Experiment Runner

The experiment runner keeps the repeated training procedure consistent across settings: user split, scaling, sequence construction, model initialization, early stopping, validation-threshold tuning, and test evaluation.

Keeping this orchestration in one place makes the comparison between experiments easier to interpret.


In [7]:
def run_experiment(name, cols, demographics=False, drop_summer_test=False):
    local_train_df, local_val_df, local_test_df, _ = seq.split_and_scale_by_users(
        df_full,
        cols,
        test_size=0.15,
        val_size=0.20,
        random_state=RANDOM_STATE,
    )
    loaders, demo_sizes = lstm.make_sequence_loaders(
        local_train_df,
        local_val_df,
        local_test_df,
        cols,
        sequence_length=SEQUENCE_LENGTH,
        batch_size=BATCH_SIZE,
        users=users_clean,
        demographics=demographics,
        drop_summer_test=drop_summer_test,
    )
    train_loader, val_loader, test_loader = loaders

    if demographics:
        model = lstm.LSTMWithDemographics(
            input_size=len(cols),
            n_gender=demo_sizes["n_gender"],
            n_canton=demo_sizes["n_canton"],
            n_class=demo_sizes["n_class"],
        ).to(device)
    else:
        model = lstm.LSTMModel(input_size=len(cols)).to(device)

    model, history = lstm.train_lstm_model(
        model,
        train_loader,
        val_loader,
        device,
        epochs=EPOCHS,
        patience=PATIENCE,
        learning_rate=LEARNING_RATE,
        demographics=demographics,
    )
    metrics, test_prob, test_y = lstm.evaluate_lstm_model(
        model,
        val_loader,
        test_loader,
        device,
        demographics=demographics,
    )

    result = {
        "experiment": name,
        "n_features": len(cols),
        "demographics": demographics,
        "drop_summer_test": drop_summer_test,
        **metrics,
    }
    return result, history, model, test_prob, test_y


## Experiments

The experiments reproduce the main questions from the older Stage 2 notebook.

- The full weekly-feature model tests the predictive value of behavioral trajectories.
- The no-summer-test variant checks whether performance is inflated by easy seasonal patterns.
- The time-only model estimates how much can be predicted from calendar information alone.
- The demographics variant tests whether profile information adds signal beyond weekly behavior.

Together these settings separate behavioral prediction from seasonal and demographic shortcuts.


In [8]:
experiments = [
    {
        "name": "all_weekly_features",
        "cols": feature_cols,
        "demographics": False,
        "drop_summer_test": False,
    },
    {
        "name": "all_weekly_features_no_summer_test",
        "cols": feature_cols,
        "demographics": False,
        "drop_summer_test": True,
    },
    {
        "name": "time_only",
        "cols": ["year", "day"],
        "demographics": False,
        "drop_summer_test": False,
    },
    {
        "name": "weekly_features_with_demographics",
        "cols": feature_cols,
        "demographics": True,
        "drop_summer_test": False,
    },
]

results = []
histories = {}
models = {}

for spec in experiments:
    print()
    print("=" * 80)
    print(spec["name"])
    result, history, model, test_prob, test_y = run_experiment(**spec)
    results.append(result)
    histories[spec["name"]] = history
    models[spec["name"]] = model
    print(result)

results_df = pd.DataFrame(results)
results_df



all_weekly_features
{'experiment': 'all_weekly_features', 'n_features': 17, 'demographics': False, 'drop_summer_test': False, 'roc_auc': 0.8836945136552299, 'f1': 0.8225779275484415, 'precision': 0.7329596636973276, 'recall': 0.9371640644996161, 'threshold': np.float64(0.3938775510204082), 'val_f1': 0.8249667352122898}

all_weekly_features_no_summer_test
{'experiment': 'all_weekly_features_no_summer_test', 'n_features': 17, 'demographics': False, 'drop_summer_test': True, 'roc_auc': 0.8375764032573269, 'f1': 0.829071012251695, 'precision': 0.7513609658815286, 'recall': 0.9247097844112769, 'threshold': np.float64(0.4591836734693878), 'val_f1': 0.8306863931632376}

time_only
{'experiment': 'time_only', 'n_features': 2, 'demographics': False, 'drop_summer_test': False, 'roc_auc': 0.8182788123458292, 'f1': 0.7968457013690889, 'precision': 0.6656224520448011, 'recall': 0.9925134374200154, 'threshold': np.float64(0.3285714285714286), 'val_f1': 0.8009650714949754}

weekly_features_with_demog

,experiment,n_features,demographics,drop_summer_test,roc_auc,f1,precision,recall,threshold,val_f1
0,all_weekly_features,17,False,False,0.883695,0.822578,0.732960,0.937164,0.393878,0.824967
1,all_weekly_features_no_summer_test,17,False,True,0.837576,0.829071,0.751361,0.924710,0.459184,0.830686
2,time_only,2,False,False,0.818279,0.796846,0.665622,0.992513,0.328571,0.800965
3,weekly_features_with_demographics,17,True,False,0.912526,0.839694,0.766899,0.927758,0.328571,0.840956


## LightGBM Baseline

We train a LightGBM model on flattened Stage 2 sequences. This baseline keeps the same user split, train-only scaling, sequence length, validation-threshold tuning, and test evaluation as the LSTM experiments, but reshapes each 12-week sequence into one tabular row.

This is useful as a non-recurrent comparison: it can use the same temporal window, but it does not model order with recurrent state.

In [9]:
def make_flat_sequences(df, cols, drop_summer_labels=False):
    x, y = seq.make_sequences(
        df,
        cols,
        sequence_length=SEQUENCE_LENGTH,
        drop_summer_labels=drop_summer_labels,
    )
    return x.reshape(len(x), -1), y


def run_lgbm_experiment(name, cols, drop_summer_test=False):
    if lgb is None:
        raise ModuleNotFoundError("LightGBM is not installed. Install `lightgbm` to run this baseline.")

    local_train_df, local_val_df, local_test_df, _ = seq.split_and_scale_by_users(
        df_full,
        cols,
        test_size=0.15,
        val_size=0.20,
        random_state=RANDOM_STATE,
    )

    x_train, y_train = make_flat_sequences(local_train_df, cols)
    x_val, y_val = make_flat_sequences(local_val_df, cols)
    x_test, y_test = make_flat_sequences(
        local_test_df,
        cols,
        drop_summer_labels=drop_summer_test,
    )

    model = lgb.LGBMClassifier(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=6,
        num_leaves=31,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=1,
        verbose=-1,
    )
    model.fit(
        x_train,
        y_train,
        eval_set=[(x_val, y_val)],
        eval_metric="auc",
        callbacks=[lgb.early_stopping(50, verbose=False)],
    )

    val_prob = model.predict_proba(x_val)[:, 1]
    test_prob = model.predict_proba(x_test)[:, 1]
    metrics = validation_tuned_binary_metrics(y_val, val_prob, y_test, test_prob)

    result = {
        "experiment": name,
        "model_type": "LightGBM",
        "n_weekly_features": len(cols),
        "n_flat_features": x_train.shape[1],
        "drop_summer_test": drop_summer_test,
        **metrics,
    }
    return result, model, test_prob, y_test


lgbm_experiments = [
    {
        "name": "all_weekly_features",
        "cols": feature_cols,
        "drop_summer_test": False,
    },
    {
        "name": "all_weekly_features_no_summer_test",
        "cols": feature_cols,
        "drop_summer_test": True,
    },
    {
        "name": "time_only",
        "cols": ["year", "day"],
        "drop_summer_test": False,
    },
]

lgbm_results = []
lgbm_models = {}

for spec in lgbm_experiments:
    print()
    print("=" * 80)
    print(f"LightGBM: {spec['name']}")
    result, model, test_prob, test_y = run_lgbm_experiment(**spec)
    lgbm_results.append(result)
    lgbm_models[spec["name"]] = model
    print(result)

lgbm_results_df = pd.DataFrame(lgbm_results)
lgbm_results_df


LightGBM: all_weekly_features


c:\Users\loicm\miniconda3\envs\dl-gpu\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\loicm\miniconda3\envs\dl-gpu\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


{'experiment': 'all_weekly_features', 'model_type': 'LightGBM', 'n_weekly_features': 17, 'n_flat_features': 204, 'drop_summer_test': False, 'roc_auc': 0.89294060652939, 'f1': 0.8313088886366217, 'precision': 0.7469400244798041, 'recall': 0.9371640644996161, 'threshold': np.float64(0.44285714285714284), 'val_f1': 0.833288485530468}

LightGBM: all_weekly_features_no_summer_test


c:\Users\loicm\miniconda3\envs\dl-gpu\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\loicm\miniconda3\envs\dl-gpu\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


{'experiment': 'all_weekly_features_no_summer_test', 'model_type': 'LightGBM', 'n_weekly_features': 17, 'n_flat_features': 204, 'drop_summer_test': True, 'roc_auc': 0.8416876319248965, 'f1': 0.8316528097175541, 'precision': 0.7485007695165313, 'recall': 0.9355887230514096, 'threshold': np.float64(0.44285714285714284), 'val_f1': 0.833288485530468}

LightGBM: time_only


c:\Users\loicm\miniconda3\envs\dl-gpu\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\loicm\miniconda3\envs\dl-gpu\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


{'experiment': 'time_only', 'model_type': 'LightGBM', 'n_weekly_features': 2, 'n_flat_features': 24, 'drop_summer_test': False, 'roc_auc': 0.8291095250606585, 'f1': 0.7982333073525592, 'precision': 0.6719447117487534, 'recall': 0.9829792679805477, 'threshold': np.float64(0.3938775510204082), 'val_f1': 0.8029451916836381}


,experiment,model_type,n_weekly_features,n_flat_features,drop_summer_test,roc_auc,f1,precision,recall,threshold,val_f1
0,all_weekly_features,LightGBM,17,204,False,0.892941,0.831309,0.746940,0.937164,0.442857,0.833288
1,all_weekly_features_no_summer_test,LightGBM,17,204,True,0.841688,0.831653,0.748501,0.935589,0.442857,0.833288
2,time_only,LightGBM,2,24,False,0.829110,0.798233,0.671945,0.982979,0.393878,0.802945


## Save Results

The result table stores the final test metrics for each experimental setting, while the history files store the training curves. These files are useful for reporting and for checking whether a result came from stable training or from a noisy run.


In [10]:
results_df.to_csv(OUTPUT_DIR / "stage2_lstm_results.csv", index=False)
lgbm_results_df.to_csv(OUTPUT_DIR / "stage2_lgbm_results.csv", index=False)
for name, history in histories.items():
    history.to_csv(OUTPUT_DIR / f"stage2_history_{name}.csv", index=False)

results_df.sort_values("f1", ascending=False)


,experiment,n_features,demographics,drop_summer_test,roc_auc,f1,precision,recall,threshold,val_f1
3,weekly_features_with_demographics,17,True,False,0.912526,0.839694,0.766899,0.927758,0.328571,0.840956
1,all_weekly_features_no_summer_test,17,False,True,0.837576,0.829071,0.751361,0.924710,0.459184,0.830686
0,all_weekly_features,17,False,False,0.883695,0.822578,0.732960,0.937164,0.393878,0.824967
2,time_only,2,False,False,0.818279,0.796846,0.665622,0.992513,0.328571,0.800965
